# Introduction

Fix $n \in \mathbb{N}.$ The $\textbf{$n$-dimensional Rubik's Cube}$ is a cube such that each of its $6$ square faces is partitioned into $n^2$ smaller colored squares, known as $\textbf{stickers}.$ Each sticker takes on a color from a set of $6$ distinct colors. An example of a cube configuration is shown below for the case $n=3.$ 
<table><tr>
<td>  <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%203x3x3%201.gif" alt="Drawing" style="width: 250px;"/> <figcaption> An Arbitrary $3$-dimensional Rubik's Cube Configuration </figcaption> </td>
</tr></table>

For any cube configuration, we can obtain $6$ possible other cube configurations; each such configuration is obtained by rotating a face $90$ degrees clockwise (the orientation is relative to how one would view the face if it were directly in front of him/her). 

Given any cube configuration, the standard challenge is to $\textbf{solve (unscramble)}$ the cube, which means to find a sequence of face rotations such that each face of the resulting configuration has squares of exactly one color (see image below). The minimal number of moves needed solve the $n$-dimensional Rubik's Cube is known as $\textbf{God's Number}.$

<table><tr>
<td> <img src="https://hlavolam.maweb.eu/images/rubikova%20kostka%203x3x3%202.gif" alt="Drawing" style="width: 250px;"/> <figcaption> A Solved $3$-dimensional Rubiks Cube Configuration </figcaption> </td>
</tr></table>


# Problem Statement

The reverse process of unscrambling, namely scrambling the Rubiks Cube, is theoretically impossible !  It is tempting to believe one can scramble the cube by performing some large number of randomly chosen face rotations on a solved configuration. But after any number of face rotations, it is always more likely to obtain certain cube configurations over other configurations. In particular, after an even number of face rotations, we show it is more likely to recover the initial solved configuration that we started with. Hence, scrambling the cube is a pointless endeavor.

# Methodology

We examine the cases $n=2$ and $n=3.$ In each case, we will first identify the $n$-dimensional Rubik's Cube with a certain permutation subgroup $H_n$ of the symmetric group of $6n^2$ letters. Then, we interpret the scrambling of the cube as a Markov chain with the underlying state space $H_n.$ For each $t \in \mathbb{N},$ we obtain the probability distribution of obtaining a state in exactly $t$ scrambles. We observe how certain states are always more likely to be obtained than others. We also measure the total variation distance between such a probability distribution and the uniform distribution in which all states are equally likely to be obtained.

$\textbf{Note}:$ In order for all code cells to run without errors, please run the code cell below, which imports all the needed libraries. If a certain library fails to import, run the magic command 
<code>!pip3 install \<insert library name here\></code> in a new cell (make sure it is a Code cell and not Markdown).

In [5]:
import numpy as np
from CubeObject import *
import scipy as sc
import scipy.sparse
import matplotlib.pyplot as plt
from sympy.combinatorics import Permutation
from sympy.combinatorics.named_groups import SymmetricGroup
import os
import json
import pandas as pd
from joblib import Parallel,delayed
dim=2

In [ ]:
def get_basic_moves(dim):
    if (dim==2):
        return ['F','L','U','FFF','LLL','UUU']
    else:
        return ['F','L','U','B','R','D']
    
def get_perm(seq,dim):
    return CubeObject(dim=dim).rotate(seq).permutation

In [ ]:
# basic_moves=get_basic_moves(dim)
# basic_perms={get_perm(seq,dim) for seq in get_basic_moves(dim)}
# all_perms=basic_perms

In [ ]:
# current_perms=all_perms
# future_perms={current_perm*basic_perm for current_perm in current_perms for basic_perm in basic_perms}
# all_perms=all_perms.union(future_perms)
# print(len(all_perms))

In [ ]:
def save_perms(perms,dim):
    directory="{}D".format(dim)
    try:
        os.makedirs(directory)
    except:
        pass
    perms_dict=dict(enumerate(perms))
    keys=perms_dict.keys()
    vals=[list(val) for val in perms_dict.values()]
    perms_dict=dict(zip(keys,vals))
    file_path=directory+'/Graph.json'
    with open(file_path, 'w') as f:
        json.dump(perms_dict, f)
    
def load_perms(dim):
    directory="{}D".format(dim)
    file_path=directory+'/Graph.json'
    if os.path.exists(file_path):
        with open(file_path,'r') as f:
            graph=json.load(f)
            vals=list(graph.values())
            
            def convert_to_perm(chunk):
                return [Permutation(l) for l in chunk]
            
            chunk_size=500000
            n_chunks=int(len(vals)/chunk_size)+1
            chunks=[vals[chunk_size*i:chunk_size*(i+1)] for i in range(n_chunks)]
            chunk_perms=Parallel(n_jobs=n_chunks,
                                 prefer='threads',
                                 verbose=100)(delayed(convert_to_perm)(chunk) for chunk in chunks)
            all_perms=[perm for chunk in chunk_perms for perm in chunk]
            return all_perms
    else:
        return set()
    
# save_perms(all_perms,dim)
all_perms=load_perms(dim)

In [ ]:
def get_rubiks_cube_subgroup(dim):
    n=6*dim**2
    S=SymmetricGroup(n)
    generators=[get_perm(seq,dim) for seq in get_basic_moves(dim)]
    H=S.subgroup(generators)
    return H

# H=get_rubiks_cube_subgroup(dim)
# H_elements=list(H.elements)    

In [ ]:
def enumeration_bijection_dicts(H_elements):
    graph=dict(enumerate(H_elements))
    reversed_graph=dict(zip(graph.values(),graph.keys()))
    return graph, reversed_graph

graph, reversed_graph=enumeration_bijection_dicts(all_perms)

In [ ]:
def save_transition_matrix_row_nonzero_column_indices_dict(transition_matrix_row_nonzero_column_indices_dict,dim):
    directory="{}D".format(dim)
    try:
        os.makedirs(directory)
    except:
        pass
    file_path=directory+"/Transition_Matrix_Row_Nonzero_Column_Indices.json"
    with open(file_path, 'w') as f:
        json.dump(transition_matrix_row_nonzero_column_indices_dict, f)
    

def get_transition_matrix_row_nonzero_column_indices_dict(dim):
    directory="{}D".format(dim)
    file_path=directory+"/Transition_Matrix_Row_Nonzero_Column_Indices.json"
    if os.path.exists(file_path):
        with open(file_path,'r') as f:
            transition_matrix_row_nonzero_column_indices_dict=json.load(f)
        return transition_matrix_row_nonzero_column_indices_dict
    else:
        return dict()


def get_transition_matrix_row_nonzero_column_indices(graph, reversed_graph,index,dim):
    current_perm=graph[index]
    attainable_perms=[current_perm * basic_perm for basic_perm in basic_perms]
    attainable_indices=[reversed_graph[attainable_perm] for attainable_perm in attainable_perms
                       if attainable_perm in reversed_graph.keys()]
    return sorted(attainable_indices)
    
def update_transition_matrix_row_nonzero_column_indices_dict(transition_matrix_row_nonzero_column_indices_dict, 
                                                        graph, 
                                                        reversed_graph,
                                                        start_index,
                                                        end_index,dim):
    for index in range(start_index,min(end_index,len(graph))):
        if index not in transition_matrix_row_nonzero_column_indices_dict.keys() or len(transition_matrix_row_nonzero_column_indices_dict[index]) != 6:
            attainable_indices=get_transition_matrix_row_nonzero_column_indices(graph, reversed_graph,index,dim)
            transition_matrix_row_nonzero_column_indices_dict.update({index : attainable_indices})
    return transition_matrix_row_nonzero_column_indices_dict

transition_matrix_row_nonzero_column_indices_dict= get_transition_matrix_row_nonzero_column_indices_dict(dim)

In [ ]:
len(transition_matrix_row_nonzero_column_indices_dict)

In [ ]:
def get_transition_matrix_coo_format(transition_matrix_row_nonzero_column_indices_dict):
    rows=np.array([key for key in transition_matrix_row_nonzero_column_indices_dict.keys()
                   for val in transition_matrix_row_nonzero_column_indices_dict[key]])

    cols=np.array([val for key in transition_matrix_row_nonzero_column_indices_dict.keys()
                   for val in transition_matrix_row_nonzero_column_indices_dict[key]])

    data=np.repeat(1/6,len(rows))
    n=len(transition_matrix_row_nonzero_column_indices_dict)
    transition_matrix=sc.sparse.coo_matrix((data, (rows, cols)), shape=(n, n))
    return transition_matrix

transition_matrix = get_transition_matrix_coo_format(transition_matrix_row_nonzero_column_indices_dict)

# Transition Matrix Analysis

In [ ]:
from scipy.sparse import linalg

In [ ]:
A=transition_matrix-sc.sparse.identity(transition_matrix.shape[0])
b=0*sc.sparse.identity(transition_matrix.shape[0]).getcol(0)
sc.sparse.linalg.spsolve(A,b)

In [ ]:
def get_chain_distribution(all_perms,transition_matrix,n_scrambles,dim):
    directory="{}D/Chain_Distributions".format(dim)
    try:
        os.makedirs(directory)
    except:
        pass
    file_path=directory+"/Chain_Dist_{}.npz".format(n_scrambles)
    
    if os.path.exists(file_path):
        return sc.sparse.load_npz(file_path)
    else:
        if(n_scrambles == 0):
            starting_index=next(i for i in range(len(all_perms)) if all_perms[i]==Permutation(size=6*dim**2))
            pi0=sc.sparse.identity(transition_matrix.shape[0]).getcol(starting_index)
            sc.sparse.save_npz(file_path,pi0)
            return pi0
        else:
            pin=transition_matrix*get_chain_distribution(all_perms,transition_matrix,n_scrambles-1,dim)
            sc.sparse.save_npz(file_path,pin)
            return pin

In [ ]:
1/order

In [ ]:
sc.sparse.find(get_chain_distribution(all_perms,transition_matrix,36,dim))[-1]

In [ ]:
def get_total_variation_distance_to_uniform(pin, dim):
    H=get_rubiks_cube_subgroup(dim)
    order=H.order()
    diffs_df=pd.DataFrame(pin.toarray().flatten()-1/order,columns=['Diff'])
    tv_distance=diffs_df[diffs_df['Diff']>=0].sum()
    return float(tv_distance)

pin=get_chain_distribution(all_perms,transition_matrix,30,dim)
tv_distance=get_total_variation_distance_to_uniform(pin, dim)
tv_distance

In [ ]:
def get_total_variation_distances_df(dim):
    directory="{}D/Chain_Distributions".format(dim)
    n=len(os.listdir(directory))

    
    file_path=directory+"/TV_Distances.csv"
    if os.path.exists(file_path):
        tv_distances_df=pd.read_csv(file_path)
    
    
    elif os.path.exists(file_path) ==False:
        
        chain_dists=[get_chain_distribution(all_perms,transition_matrix,n_scrambles,dim) 
                     for n_scrambles in range(n)]
        tv_distances=[get_total_variation_distance_to_uniform(pin,dim) for pin in chain_dists]
        
        tv_distances_df=pd.DataFrame(tv_distances,columns=['Total_Variation_Distance'])
        tv_distances_df.to_csv(file_path,index=False)
    

    if (n-2 != len(tv_distances_df)):
        new_chain_dists=[get_chain_distribution(all_perms,transition_matrix,n_scrambles,dim) 
                     for n_scrambles in range(len(tv_distances_df), n)]
        new_tv_distances=[get_total_variation_distance_to_uniform(pin,dim) for pin in new_chain_dists]
        
        new_tv_distances_df=pd.DataFrame(new_tv_distances,columns=['Total_Variation_Distance'])
        tv_distances_df=pd.concat([tv_distances_df,new_tv_distances_df],axis=0)
        tv_distances_df=tv_distances_df.reset_index().drop(columns='index')
        tv_distances_df.to_csv(file_path,index=False)
    return tv_distances_df

# get_chain_distribution(transition_matrix,10,dim)
total_variation_distances_df=get_total_variation_distances_df(dim)
fig,ax=plt.subplots(1,figsize=(20,5))
ax.scatter(total_variation_distances_df.index,total_variation_distances_df['Total_Variation_Distance'])
ax.set_title(str(dim) +'D Chain Distribution Total Variation Distances after n Scrambles')
ax.set_xlabel('Number of Scrambles')
ax.set_ylabel('Total Variation Distance')

# References

- https://www.lifehacker.com.au/2020/01/how-hard-is-it-to-scramble-rubiks-cube/
- http://anttila.ca/michael/devilsalgorithm/#:~:text=Definition%3A%20A%20Devil's%20Algorithm%20is,in%20the%20shortest%20Devil's%20Algorithm.